<a href="https://colab.research.google.com/github/PoojaEsh/Interactive-Data-Exploration-and-Visualization-System/blob/main/itcfnlproj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

!pip install xgboost gradio joblib openpyxl -q

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

DATA_FILE = list(uploaded.keys())[0]
TARGET_COL = "Output_Status"
BATCH_SIZE = 10

df = pd.read_excel(DATA_FILE)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [
    "Machine_ID", "Shift", "Cigarette_Type", "Blend_Type",
    "Moisture_Level", "Tobacco_Density", "Cigarette_Weight",
    "Burn_Rate", "Nicotine_Content", "Ash_Content", "Output_Status"
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

target_le = LabelEncoder()
df[TARGET_COL] = target_le.fit_transform(df[TARGET_COL].astype(str))

encoders = {}
categorical_cols = ["Machine_ID", "Shift", "Cigarette_Type", "Blend_Type"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# Exclude Machine_ID to reduce leakage and make comparison stricter
feature_cols = [
    col for col in df.columns
    if col != TARGET_COL and col != "Machine_ID"
]

print("Feature columns:", feature_cols)
print("Target classes:", list(target_le.classes_))

def build_feature_vector(batch_df, feature_cols):
    features = []
    for col in feature_cols:
        col_data = batch_df[col]
        features.append(col_data.mean())
        features.append(col_data.std(ddof=0))
        features.append(col_data.min())
        features.append(col_data.max())
    return features

def create_batches(df, target_col, feature_cols, batch_size=10):
    X_batches = []
    y_batches = []

    for i in range(0, len(df) - batch_size + 1, batch_size):
        batch = df.iloc[i:i + batch_size]
        features = build_feature_vector(batch, feature_cols)
        target_value = batch[target_col].mode()[0]
        X_batches.append(features)
        y_batches.append(target_value)

    return np.array(X_batches), np.array(y_batches)

X, y = create_batches(df, TARGET_COL, feature_cols, BATCH_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=80,
        learning_rate=0.08,
        max_depth=3,
        random_state=42
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_split=6,
        min_samples_leaf=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=800,
        max_depth=10,
        learning_rate=0.02,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.5,
        reg_alpha=0.2,
        min_child_weight=1,
        random_state=42,
        eval_metric="logloss"
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nCross-Validation Accuracy:")
cv_scores = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    cv_scores[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f} (std: {scores.std():.4f})")

best_model = models["XGBoost"]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print("\nFinal Deployment Model: XGBoost")
print(f"XGBoost CV Accuracy: {cv_scores['XGBoost']:.4f}")
print(f"XGBoost Test Accuracy: {test_acc:.4f}")
print("Note: Machine_ID was excluded from training features to reduce leakage and make model comparison stricter.")
print("All 5 models were compared honestly, and XGBoost was selected as the final deployment model.")

joblib.dump(best_model, "best_model.pkl")
joblib.dump(encoders, "encoders.pkl")
joblib.dump(target_le, "target_encoder.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
joblib.dump(cv_scores, "model_scores.pkl")
df.to_csv("final_dataset.csv", index=False)

print("\nSaved files successfully.")
print("Saved files: best_model.pkl, encoders.pkl, target_encoder.pkl, feature_cols.pkl, model_scores.pkl, final_dataset.csv")


Saving cigarette_dataset_2500rows_with_shifts.xlsx to cigarette_dataset_2500rows_with_shifts.xlsx
Dataset shape: (2500, 11)
Columns: ['Machine_ID', 'Shift', 'Cigarette_Type', 'Blend_Type', 'Moisture_Level', 'Tobacco_Density', 'Cigarette_Weight', 'Burn_Rate', 'Nicotine_Content', 'Ash_Content', 'Output_Status']
Feature columns: ['Shift', 'Cigarette_Type', 'Blend_Type', 'Moisture_Level', 'Tobacco_Density', 'Cigarette_Weight', 'Burn_Rate', 'Nicotine_Content', 'Ash_Content']
Target classes: ['DEFECTIVE', 'SAFE']
X shape: (250, 36)
y shape: (250,)

Cross-Validation Accuracy:
Logistic Regression: 0.9700 (std: 0.0100)
KNN: 0.9700 (std: 0.0100)
Gradient Boosting: 0.9750 (std: 0.0224)
Extra Trees: 0.9700 (std: 0.0100)
XGBoost: 0.9850 (std: 0.0200)

Final Deployment Model: XGBoost
XGBoost CV Accuracy: 0.9850
XGBoost Test Accuracy: 1.0000
Note: Machine_ID was excluded from training features to reduce leakage and make model comparison stricter.
All 5 models were compared honestly, and XGBoost was s

In [ ]:
print("Machine options:", list(encoders["Machine_ID"].classes_))
print("Shift options:", list(encoders["Shift"].classes_))
print("Cigarette type options:", list(encoders["Cigarette_Type"].classes_))


Machine options: ['M1', 'M10', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
Shift options: ['E', 'L', 'N']
Cigarette type options: ['Aromatic', 'Clove Cigarette', 'Filtered', 'Hand Rolled', 'King Size', 'Light', 'Menthol', 'Premium', 'Slim', 'Unfiltered']


In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

TARGET_COL = "Output_Status"
MACHINE_COL = "Machine_ID"
SHIFT_COL = "Shift"
CIGAR_COL = "Cigarette_Type"
BLEND_COL = "Blend_Type"
BATCH_SIZE = 10

model = joblib.load("best_model.pkl")
encoders = joblib.load("encoders.pkl")
target_le = joblib.load("target_encoder.pkl")
feature_cols = joblib.load("feature_cols.pkl")
model_scores = joblib.load("model_scores.pkl")
df = pd.read_csv("final_dataset.csv")

machine_options = list(encoders[MACHINE_COL].classes_)
shift_options = list(encoders[SHIFT_COL].classes_)
cigar_options = list(encoders[CIGAR_COL].classes_)
blend_options = list(encoders[BLEND_COL].classes_)

custom_css = """
* {
    font-family: "Times New Roman", Times, serif !important;
}
.gradio-container {
    font-family: "Times New Roman", Times, serif !important;
}
"""

def decode_dataframe(data):
    decoded = data.copy()
    decoded[MACHINE_COL] = encoders[MACHINE_COL].inverse_transform(decoded[MACHINE_COL].astype(int))
    decoded[SHIFT_COL] = encoders[SHIFT_COL].inverse_transform(decoded[SHIFT_COL].astype(int))
    decoded[CIGAR_COL] = encoders[CIGAR_COL].inverse_transform(decoded[CIGAR_COL].astype(int))
    decoded[BLEND_COL] = encoders[BLEND_COL].inverse_transform(decoded[BLEND_COL].astype(int))
    decoded[TARGET_COL] = target_le.inverse_transform(decoded[TARGET_COL].astype(int))
    return decoded

decoded_full_df = decode_dataframe(df)

def make_result_card(result, confidence, rows_used):
    if result == "SAFE":
        bg = "#e8f7ee"
        border = "#22a06b"
        title_text = "#14532d"
        badge = "#22a06b"
        value_text = "#0f172a"
    else:
        bg = "#fdecec"
        border = "#d92d20"
        title_text = "#7a271a"
        badge = "#d92d20"
        value_text = "#0f172a"

    return f"""
    <div style="
        background:{bg};
        border:2px solid {border};
        border-radius:16px;
        padding:22px 24px;
        box-shadow:0 6px 18px rgba(0,0,0,0.08);
        font-family:'Times New Roman', Times, serif;
    ">
        <div style="
            display:inline-block;
            background:{badge};
            color:white;
            font-weight:700;
            padding:8px 16px;
            border-radius:999px;
            font-size:18px;
            margin-bottom:14px;
        ">
            {result}
        </div>

        <div style="
            font-size:30px;
            font-weight:800;
            color:{title_text};
            margin-bottom:12px;
        ">
            Prediction Result
        </div>

        <div style="font-size:20px; color:{value_text}; margin-bottom:8px;">
            Confidence: <b style="color:{value_text};">{confidence:.2f}%</b>
        </div>

        <div style="font-size:20px; color:{value_text};">
            Rows used: <b style="color:{value_text};">{rows_used}</b>
        </div>
    </div>
    """

def plot_last_rows(display_data):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.patch.set_facecolor("white")

    x = list(range(1, len(display_data) + 1))
    metrics = [
        ("Moisture_Level", "#1f77b4"),
        ("Cigarette_Weight", "#ff7f0e"),
        ("Burn_Rate", "#2ca02c"),
        ("Nicotine_Content", "#d62728"),
    ]

    for ax, (col, color) in zip(axes.flatten(), metrics):
        ax.plot(x, display_data[col], marker="o", linewidth=2.5, color=color)
        ax.set_title(col.replace("_", " "), fontsize=11, fontweight="bold", fontname="Times New Roman")
        ax.set_xlabel("Recent Record", fontname="Times New Roman")
        ax.set_ylabel("Value", fontname="Times New Roman")
        ax.grid(True, alpha=0.3)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontname("Times New Roman")

    plt.suptitle("Last Selected Rows - Sensor Trends", fontsize=15, fontweight="bold", fontname="Times New Roman")
    plt.tight_layout()
    return fig

def plot_machine_defect_trend():
    trend_df = decoded_full_df.copy()
    trend_df["Defective_Flag"] = (trend_df[TARGET_COL] == "DEFECTIVE").astype(int)

    summary = (
        trend_df.groupby(MACHINE_COL)["Defective_Flag"]
        .mean()
        .reset_index()
        .sort_values("Defective_Flag", ascending=False)
    )
    summary["Defect_Percent"] = summary["Defective_Flag"] * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(summary[MACHINE_COL], summary["Defect_Percent"], color="#d92d20", alpha=0.85)

    ax.set_title("Machine-wise Defect Trend", fontsize=15, fontweight="bold", fontname="Times New Roman")
    ax.set_xlabel("Machine", fontname="Times New Roman")
    ax.set_ylabel("Defect Rate (%)", fontname="Times New Roman")
    ax.set_ylim(0, max(summary["Defect_Percent"].max() + 5, 10))
    ax.grid(axis="y", alpha=0.25)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname("Times New Roman")

    for bar, value in zip(bars, summary["Defect_Percent"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            fontname="Times New Roman"
        )

    plt.tight_layout()
    return fig

def plot_model_comparison():
    names = list(model_scores.keys())
    values = list(model_scores.values())

    colors = []
    for name in names:
        if name == "XGBoost":
            colors.append("#16a34a")
        else:
            colors.append("#94a3b8")

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, values, color=colors)

    ax.set_title("Model Comparison Accuracy", fontsize=15, fontweight="bold", fontname="Times New Roman")
    ax.set_ylabel("Accuracy", fontname="Times New Roman")
    ax.set_ylim(min(values) - 0.03, max(values) + 0.03)
    ax.grid(axis="y", alpha=0.25)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname("Times New Roman")
    plt.xticks(rotation=20)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{value:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            fontname="Times New Roman"
        )

    plt.tight_layout()
    return fig

def predict_quality(machine_name, shift_name, cigar_type, blend_type):
    machine_encoded = encoders[MACHINE_COL].transform([machine_name])[0]
    shift_encoded = encoders[SHIFT_COL].transform([shift_name])[0]
    cigar_encoded = encoders[CIGAR_COL].transform([cigar_type])[0]
    blend_encoded = encoders[BLEND_COL].transform([blend_type])[0]

    filtered_data = df[
        (df[MACHINE_COL] == machine_encoded) &
        (df[SHIFT_COL] == shift_encoded) &
        (df[CIGAR_COL] == cigar_encoded) &
        (df[BLEND_COL] == blend_encoded)
    ].tail(BATCH_SIZE)

    model_fig = plot_model_comparison()

    if len(filtered_data) == 0:
        empty_fig1, ax1 = plt.subplots(figsize=(8, 4))
        ax1.text(0.5, 0.5, "No matching rows found", ha="center", va="center", fontsize=16, fontname="Times New Roman")
        ax1.axis("off")

        empty_fig2, ax2 = plt.subplots(figsize=(8, 4))
        ax2.text(0.5, 0.5, "Trend unavailable", ha="center", va="center", fontsize=16, fontname="Times New Roman")
        ax2.axis("off")

        return (
            """
            <div style="background:#fff4e5;border:2px solid #f59e0b;border-radius:16px;padding:18px;font-family:'Times New Roman', Times, serif;">
                <div style="font-size:22px;font-weight:800;color:#92400e;">No matching rows found</div>
                <div style="font-size:16px;color:#78350f;margin-top:8px;">Try a different combination of filters.</div>
            </div>
            """,
            empty_fig1,
            empty_fig2,
            model_fig
        )

    features = []
    for col in feature_cols:
        col_data = filtered_data[col]
        features.append(col_data.mean())
        features.append(col_data.std(ddof=0))
        features.append(col_data.min())
        features.append(col_data.max())

    features = np.array(features).reshape(1, -1)

    pred = model.predict(features)[0]
    prob = model.predict_proba(features)[0].max() * 100
    result = target_le.inverse_transform([int(pred)])[0]

    display_data = decode_dataframe(filtered_data)

    card_html = make_result_card(result, prob, len(display_data))
    last_rows_chart = plot_last_rows(display_data)
    defect_trend_chart = plot_machine_defect_trend()

    return card_html, last_rows_chart, defect_trend_chart, model_fig

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Cigarette Machine Quality Predictor",
    css=custom_css
) as demo:
    gr.Markdown("""
    # Cigarette Machine Quality Predictor
    Professional quality prediction dashboard powered by XGBoost.
    Select machine, shift, cigarette type, and blend type to view prediction results and performance trends.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            machine_input = gr.Dropdown(choices=machine_options, label="Machine", value=machine_options[0])
            shift_input = gr.Dropdown(choices=shift_options, label="Shift", value=shift_options[0])
            cigar_input = gr.Dropdown(choices=cigar_options, label="Cigarette Type", value=cigar_options[0])
            blend_input = gr.Dropdown(choices=blend_options, label="Blend Type", value=blend_options[0])
            predict_btn = gr.Button("Predict Quality", variant="primary")

        with gr.Column(scale=2):
            result_card = gr.HTML(label="Prediction")

    with gr.Row():
        last_rows_plot = gr.Plot(label="Last Selected Rows")
        machine_trend_plot = gr.Plot(label="Machine-wise Defect Trend")

    with gr.Row():
        model_comparison_plot = gr.Plot(label="Model Comparison")

    predict_btn.click(
        fn=predict_quality,
        inputs=[machine_input, shift_input, cigar_input, blend_input],
        outputs=[result_card, last_rows_plot, machine_trend_plot, model_comparison_plot]
    )

demo.launch(share=True)


/tmp/ipykernel_4307/4272869060.py:261: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4307/4272869060.py:261: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39329dfccd99d5dce3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
!git config --global user.name "PoojEsh"
!git config --global user.email "poojae028@gmail.com"

In [6]:
!git clone https://github.com/PoojEsh/cigarette-quality-prediction-system.git

Cloning into 'cigarette-quality-prediction-system'...
fatal: could not read Username for 'https://github.com': No such device or address
